In [2]:
import numpy as np
from IPython.display import clear_output 


MODO_TEST = True 
jugar = 0
seleccion = 0
orientacion = 0
letras = np.array(list("ABCDEFGHIJ"))
numeros = np.arange(1, 11).astype(str)

TITULO = r"""
    █   █ █   █ █   █ ████  █████ ████
    █   █ █   █ ██  █ █   █   █   █   █
    █████ █   █ █ █ █ █   █   █   ████
    █   █ █   █ █  ██ █   █   █   █  █
    █   █  ███  █   █ ████  █████ █   █

█      ███    █████ █      ███  █████  ███
█     █   █   █     █     █   █   █   █   █
█     █████   ████  █     █   █   █   █████
█     █   █   █     █     █   █   █   █   █
█████ █   █   █     █████  ███    █   █   █
                 ___|___
             ___|_______|___
      ______|_______________|______
      \   o    o    o    o    o   /
  ~~~~~\_________________________/~~~~~
     ~~~      ~~~~       ~~~      ~~~
  ╔══════════════╗      ╔══════════════╗
  ║  [1] JUGAR   ║      ║  [2] SALIR   ║
  ╚══════════════╝      ╚══════════════╝
"""
ORIENTACION3 = """
    [H] HORIZONTAL:        [V] VERTICAL:
                                   █
         █ █ █                     █
                                   █
"""

ORIENTACION4 = """
    [H] HORIZONTAL:        [V] VERTICAL:
                                   █
         █ █ █ █                   █
                                   █
                                   █
"""
ORIENTACION2 = """
    [H] HORIZONTAL:        [V] VERTICAL:
                                   █
         █ █                       █
                                   
"""
ORIENTACION5 = """
    [H] HORIZONTAL:        [V] VERTICAL:
                                   █
        █ █ █ █ █                  █
                                   █
                                   █
                                   █
"""



tablero = np.full((10,10), 0)
tablero_ataque = np.char.add(letras[:, None], numeros[None, :])

tablero_rival = np.full((10, 10), 0)
tablero_rival_ataque = np.char.add(letras[:, None], numeros[None, :])

barcos = {
    "Portaaviones": 5,
    "Acorazado": 4,
    "Destructor": 3,
    "Submarino": 3,
    "Patrullero": 2
}
lista_barcos_faltantes = list(barcos.keys())
lista_seleccion = [1,2,3,4,5]
lista_orientacion = ["h", "v"]

'''
tablero = np.array([
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 1, 1, 1, 1, 1, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 1, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 1, 0],
    [0, 1, 1, 1, 0, 0, 0, 0, 1, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 1, 0],
    [0, 0, 0, 0, 1, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 1, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 1, 0, 0, 0, 1, 1],
])
'''

tablero_posiciones = np.char.add(letras[:, None], numeros[None, :])

barcos_jugador = {}    
barcos_rival = {}


def tachar(texto: str)->str:
    return f"\033[9m{texto}\033[0m"

def lista_barcos(barcos: dict,barcos_faltantes:list):
    indice = 5
    print('Barcos por Colocar:')
    for nombre, tamano in barcos.items():
        texto = f"{nombre} ({tamano})casillas"
        if nombre in barcos_faltantes:
            print(f"    [{indice}] {texto}")
        else:
            print("    "+tachar(f"[{indice}] {texto}"))
        indice -=1


def matriz_posibles(tamano, orientacion, tablero):
    lado = tablero.shape[0]
    matriz = np.zeros((lado,lado), dtype = int)
 
    if orientacion in ["H", "h"]:
        for i in range(lado - tamano + 1): 
            matriz[i:i+tamano, i] = 1
    else:
            for i in range(lado - tamano + 1):
                matriz[i, i:i+tamano] = 1
    return matriz

    
def combinacion_posibles(tablero, combinaciones_posibles, orientacion, tamano):
    N = tablero.shape[0]
    limite = N - (tamano -1)

    if orientacion in ["H", "h"]:
        posibles = tablero @ combinaciones_posibles == 0
        posibles[:, limite:] = False
    else:
        posibles = combinaciones_posibles @ tablero == 0
        posibles[limite:,:] = False
    return posibles

def coloca_barco(tamano, orientacion, posicion, tablero):
    fila = ord(posicion[0].upper()) - ord("A")
    col = int(posicion[1:]) - 1

    if orientacion.lower() == "h":
        tablero[fila, col:col + tamano] = 1
        casillas = [(fila, c) for c in range(col, col + tamano)]
    else:
        tablero[fila:fila + tamano, col] = 1
        casillas = [(f, col) for f in range(fila, fila + tamano)]

    return casillas

def imprime_tablero_barcos(tablero):
    return np.where(tablero == 1, "O", "~")

def coloca_barcos_aleatorio(barcos, tablero):
    for nombre, tamano in barcos.items():
        valores = []
        while len(valores) == 0:
            orientacion = np.random.choice(["h", "v"])
            m = matriz_posibles(tamano, orientacion, tablero)
            j = combinacion_posibles(tablero, m, orientacion, tamano)
            valores = tablero_posiciones[j]
        posicion = np.random.choice(valores)
        barcos_rival[nombre] = coloca_barco(tamano, orientacion, posicion, tablero)

def coordenada_a_indices(posicion):
    fila = ord(posicion[0].upper()) - ord("A")
    col = int(posicion[1:]) - 1
    return fila, col

def barco_tocado(fila, col, barcos_colocados):
    for nombre, casillas in barcos_colocados.items():
        if (fila, col) in casillas:
            return nombre


def barco_hundido(nombre, barcos_colocados, tablero_simbolos):
    for fila, col in barcos_colocados[nombre]:
        if tablero_simbolos[fila, col] != "X":
            return False
    return True


def disparar(posicion, tablero_simbolos, tablero_registro, barcos_colocados):
    fila, col = coordenada_a_indices(posicion)

    if tablero_simbolos[fila, col] == "O":
        tablero_simbolos[fila, col] = "X"
        tablero_registro[fila, col] = "X"
        nombre = barco_tocado(fila, col, barcos_colocados)
        if barco_hundido(nombre, barcos_colocados, tablero_simbolos):
            return "hundido", nombre
        return "tocado", nombre
    else:
        tablero_simbolos[fila, col] = "-"
        tablero_registro[fila, col] = "-"
        return "agua", None


def quedan_barcos(tablero_simbolos):
    return np.any(tablero_simbolos == "O")


def mensaje_disparo(quien, posicion, resultado, nombre):
    if resultado == "agua":
        return f"{quien} en {posicion}: AGUA (-)"
    elif resultado == "tocado":
        return f"{quien} en {posicion}: TOCADO (X)"
    else:
        return f"{quien} en {posicion}: TOCADO Y HUNDIDO -> {nombre}"


def coordenada_a_indices(posicion):
    fila = ord(posicion[0].upper()) - ord("A")
    col = int(posicion[1:]) - 1
    return fila, col



In [3]:
print(TITULO)
while jugar not in ["1","2"]:
    jugar = input("Selecciona [1] para jugar o [2] para salir...")
    print("-" * 45)


while jugar == "1" and lista_barcos_faltantes:

    while seleccion not in lista_seleccion:
        clear_output()
        print(imprime_tablero_barcos(tablero))
        print("-" * 45)
        lista_barcos(barcos, lista_barcos_faltantes)
        texto = input("¿Qué barco quieres colocar?: ").strip()
        if texto in ["1", "2", "3", "4", "5"]:
            seleccion = int(texto)

    if seleccion >= 3:
        tamano = seleccion
    elif seleccion == 2:
        tamano = 3
    else:
        tamano = 2

    

    while orientacion not in lista_orientacion:
        clear_output()

        print(imprime_tablero_barcos(tablero))
        print("-" * 45)

        if tamano == 5:
            print(ORIENTACION5)
        elif tamano == 4:
            print(ORIENTACION4)
        elif tamano == 3:
            print(ORIENTACION3)
        else:
            print(ORIENTACION2)
         
        orientacion = input('Orientacion: ').strip().lower()

        

        if orientacion in lista_orientacion:
            m = matriz_posibles(tamano, orientacion, tablero)
            j = combinacion_posibles(tablero, m, orientacion,tamano)
            tablero_visible = np.where(j, tablero_posiciones, "X")
            print(tablero_visible)
    
    valores = set(tablero_posiciones[j].tolist())

    while True:
        clear_output()
        print(tablero_visible) 
        posicion_inicial = input("donde quieres poner el barco: ").strip().upper()
        if posicion_inicial in valores:
            break

    barcos_jugador[list(barcos.keys())[5 - seleccion]] = coloca_barco(tamano, orientacion,posicion_inicial, tablero)

    print("-" * 45)
    
    tamano = 0
    orientacion = 0
    lista_barcos_faltantes.remove(list(barcos.keys())[5 - seleccion]) 
    lista_seleccion.remove(seleccion)                                 
    seleccion = 0
    clear_output()
    print('     TABLERO:')
    print(imprime_tablero_barcos(tablero))
    

if jugar == "1":

    coloca_barcos_aleatorio(barcos, tablero_rival)

    tablero_jugador = imprime_tablero_barcos(tablero)              
    tablero_rival_simbolos = imprime_tablero_barcos(tablero_rival)  

    disparos_jugador = set(tablero_posiciones.flatten().tolist())
    disparos_maquina = tablero_posiciones.flatten().tolist()
    mensaje_jugador = ""
    mensaje_maquina = ""
    ganador = None

    while ganador is None:

        resultado = "tocado"
        while resultado != "agua" and ganador is None:

            while True:
                clear_output()
                print('     TU TABLERO:')
                print(tablero_jugador)
                print("-" * 45)
                print('     TABLERO ATAQUE:')
                print(tablero_ataque)
                print("-" * 45)
                if MODO_TEST:                                   # NUEVO
                    print('     TABLERO RIVAL (TEST):')         # NUEVO
                    print(tablero_rival_simbolos)               # NUEVO
                    print("-" * 45) 
                print(mensaje_jugador)
                print(mensaje_maquina)
                disparo = input("¿Dónde quieres disparar?: ").strip().upper()
                if disparo in disparos_jugador:
                    break

            disparos_jugador.remove(disparo)
            resultado, nombre = disparar(disparo, tablero_rival_simbolos, tablero_ataque, barcos_rival)
            mensaje_jugador = mensaje_disparo("Tu disparo", disparo, resultado, nombre)

            if not quedan_barcos(tablero_rival_simbolos):
                ganador = "jugador"
            elif resultado != "agua":
                mensaje_jugador += "  ->  ¡Vuelves a disparar!"

        if ganador is not None:
            break

        mensaje_maquina = ""
        resultado = "tocado"
        while resultado != "agua" and ganador is None:
            disparo_maquina = str(np.random.choice(disparos_maquina))
            disparos_maquina.remove(disparo_maquina)
            resultado, nombre = disparar(disparo_maquina, tablero_jugador, tablero_rival_ataque, barcos_jugador)
            mensaje_maquina += mensaje_disparo("La máquina dispara", disparo_maquina, resultado, nombre) + "\n"

            if not quedan_barcos(tablero_jugador):
                ganador = "maquina"

   
    clear_output()
    print("=" * 45)
    if ganador == "jugador":
        print("     ¡HAS GANADO! Hundiste toda la flota rival")
    else:
        print("     HAS PERDIDO. La máquina hundió tu flota")
    print("=" * 45)
    print(mensaje_jugador)
    print(mensaje_maquina)
    print("-" * 45)
    print('     TU TABLERO:')
    print(tablero_jugador)
    print("-" * 45)
    print('     TABLERO RIVAL:')
    print(tablero_rival_simbolos)
    
        
        

    
    

     ¡HAS GANADO! Hundiste toda la flota rival
Tu disparo en J9: TOCADO Y HUNDIDO -> Portaaviones
La máquina dispara en B2: AGUA (-)

---------------------------------------------
     TU TABLERO:
[['~' '~' '~' '~' 'O' 'O' 'O' '~' '~' '~']
 ['~' '-' '~' '~' '~' '~' '~' '~' '~' '~']
 ['~' 'O' '~' '~' '~' '~' '~' '~' '~' '~']
 ['~' 'O' '~' '~' '~' '~' '~' '~' '~' '~']
 ['~' 'O' '~' '~' '~' '~' '~' '~' '~' '~']
 ['~' 'O' '~' '~' '~' '~' '~' '~' '~' '~']
 ['~' 'O' 'O' '~' '~' '~' '~' 'O' '~' '~']
 ['~' '~' 'O' '~' '~' '~' '~' 'O' '~' '~']
 ['~' '~' 'O' '~' '~' '~' '~' 'O' '~' '~']
 ['O' 'O' 'O' '~' '~' '~' '~' '~' '~' '~']]
---------------------------------------------
     TABLERO RIVAL:
[['~' '~' '~' '~' 'X' '~' '~' '~' '~' '~']
 ['~' '~' '~' '~' 'X' '~' 'X' '~' '~' '~']
 ['~' '~' '~' '~' 'X' '~' 'X' '~' '~' '~']
 ['~' '~' '~' '~' '~' '~' 'X' '~' '~' '~']
 ['~' '~' '~' '~' '~' '~' '~' '~' '~' '~']
 ['~' '~' 'X' 'X' 'X' 'X' 'X' 'X' 'X' '~']
 ['~' '~' '~' '~' '~' '~' '~' '~' 'X' '-']
 ['~'

In [5]:
m = matriz_posibles(5, "V", tablero)
j = combinacion_posibles(tablero, m, "V",5)

print(j.size)
dami = np.arange(1,101)
print(j)
print(dami.size)
dami = dami.reshape(10,10)
vick = dami* j
print(vick)


100
[[ True False  True  True False False False  True  True  True]
 [ True False  True  True  True  True  True  True  True  True]
 [ True False False  True  True  True  True False  True  True]
 [ True False False  True  True  True  True False  True  True]
 [ True False False  True  True  True  True False  True  True]
 [False False False  True  True  True  True False  True  True]
 [False False False False False False False False False False]
 [False False False False False False False False False False]
 [False False False False False False False False False False]
 [False False False False False False False False False False]]
100
[[ 1  0  3  4  0  0  0  8  9 10]
 [11  0 13 14 15 16 17 18 19 20]
 [21  0  0 24 25 26 27  0 29 30]
 [31  0  0 34 35 36 37  0 39 40]
 [41  0  0 44 45 46 47  0 49 50]
 [ 0  0  0 54 55 56 57  0 59 60]
 [ 0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  0]]


In [6]:
m = matriz_posibles(5, "V", tablero)
j = combinacion_posibles(tablero, m, "V",5)

print(j.size)
dami = np.arange(1,101)
print(j)
print(dami.size)
dami = dami.reshape(10,10)
vick = tablero_posiciones* j
print(vick)


100
[[ True False  True  True False False False  True  True  True]
 [ True False  True  True  True  True  True  True  True  True]
 [ True False False  True  True  True  True False  True  True]
 [ True False False  True  True  True  True False  True  True]
 [ True False False  True  True  True  True False  True  True]
 [False False False  True  True  True  True False  True  True]
 [False False False False False False False False False False]
 [False False False False False False False False False False]
 [False False False False False False False False False False]
 [False False False False False False False False False False]]
100


UFuncTypeError: ufunc 'multiply' did not contain a loop with signature matching types (dtype('<U22'), dtype('bool')) -> None